In [1]:
import os
from dotenv import load_dotenv
from langchain_google_genai import ChatGoogleGenerativeAI

# Load environment variables (.env file should contain GOOGLE_API_KEY)
load_dotenv()

# Initialize Gemini Model
chat_model = ChatGoogleGenerativeAI(
    model="gemini-flash-lite-latest", 
    temperature=0.7 # کمی خلاقیت برای بازنویسی ادبی
)

print("✅ Gemini model initialized successfully!")

✅ Gemini model initialized successfully!


In [3]:
from langchain_core.prompts import PromptTemplate
from langchain_core.example_selectors import LengthBasedExampleSelector

# تعریف مثال‌های ادبی
examples = [
    {"input": "خوشحالم", "output": "در پوست خود نمی‌گنجم و سرشار از شعف هستم."},
    {"input": "هوا سرد است", "output": "سرمای هوا تا مغز استخوان نفوذ می‌کند و زمستان رخ نشان می‌دهد."},
    {"input": "خسته نباشید", "output": "امیدوارم همواره پرانرژی باشید و خستگی را احساس نکنید."},
    {"input": "شب بخیر", "output": "آرزوی شبی آرام و پرستاره برای شما دارم."}
]

# قالبی برای محاسبه طول مثال‌ها
length_prompt = PromptTemplate.from_template("Input: {input} Output: {output}")

# ساخت انتخاب‌گر (مجموع طول ورودی + مثال‌ها نباید از 40 تجاوز کند)
example_selector = LengthBasedExampleSelector(
    examples=examples,
    example_prompt=length_prompt,
    max_length=40,
)

print("✅ Length-Based Example Selector is ready!")

✅ Length-Based Example Selector is ready!


In [6]:
short_input = {"input": "سلام"}
selected = example_selector.select_examples(short_input)

print(f"تعداد مثال‌های انتخاب شده: {len(selected)}")
for ex in selected:
    print(ex)

تعداد مثال‌های انتخاب شده: 2
{'input': 'خوشحالم', 'output': 'در پوست خود نمی\u200cگنجم و سرشار از شعف هستم.'}
{'input': 'هوا سرد است', 'output': 'سرمای هوا تا مغز استخوان نفوذ می\u200cکند و زمستان رخ نشان می\u200cدهد.'}


In [7]:
long_input = {"input": "امروز صبح که از خواب بیدار شدم احساس کردم که هوا بسیار عالی است و می‌خواهم به کوه بروم و از طبیعت لذت ببرم..."}
selected = example_selector.select_examples(long_input)

print(f"تعداد مثال‌های انتخاب شده: {len(selected)}")
for ex in selected:
    print(ex)

تعداد مثال‌های انتخاب شده: 1
{'input': 'خوشحالم', 'output': 'در پوست خود نمی\u200cگنجم و سرشار از شعف هستم.'}


In [8]:
from langchain_core.prompts import ChatPromptTemplate, FewShotChatMessagePromptTemplate

# 1. تعریف قالب چت برای هر مثال
example_prompt = ChatPromptTemplate.from_messages([
    ("human", "{input}"), 
    ("ai", "{output}")
])

# 2. ساخت قالب پویای چندنمونه‌ای با انتخاب‌گر طولی
dynamic_prompt = FewShotChatMessagePromptTemplate(
    example_selector=example_selector,
    example_prompt=example_prompt,
)

# 3. ترکیب پیام سیستمی، مثال‌های پویا و ورودی کاربر
final_prompt = ChatPromptTemplate.from_messages([
    ("system", "جملات را به صورت ادبی بازنویسی کن."),
    dynamic_prompt,
    ("human", "{input}"),
])

# 4. ساخت زنجیره
chain = final_prompt | chat_model

print("✅ Pipeline is ready to run!")

✅ Pipeline is ready to run!


In [9]:
text = "من موسیقی دوست دارم."

print("⏳ Generating literary rewrite...\n")
response = chain.invoke({"input": text})

print("--- 🤖 Gemini Response ---")
print(response.content)

print("\n--- 🔍 What was actually sent to the model? ---")
print(final_prompt.format(input=text))

⏳ Generating literary rewrite...



Direct use of automatic function calling (AFC) in Models.generate_content is not recommended. Instead, we recommend to use AFC in Chat.send_message. Similarly, direct use of AFC in Models.generate_content_stream is not recommended. Instead, we recommend to use AFC in Chat.send_message_stream.


--- 🤖 Gemini Response ---
[{'type': 'text', 'text': 'روح من با نغمه\u200cها و آواهای موسیقی آمیخته است و به آن عشق می\u200cورزد.', 'extras': {'signature': 'EmAKXgFpFH0T4BydHnCwiEj4798n6f51Mz/1P8wOLf9RpF+N8xlngxE96whyOOGow6nrC7cq7GCE2ntkfa0S6wuLttiHM72gYikrO69z3HRmIo5CZjFfDTXxKo+NMhYfe9w='}}]

--- 🔍 What was actually sent to the model? ---
System: جملات را به صورت ادبی بازنویسی کن.
Human: خوشحالم
AI: در پوست خود نمی‌گنجم و سرشار از شعف هستم.
Human: هوا سرد است
AI: سرمای هوا تا مغز استخوان نفوذ می‌کند و زمستان رخ نشان می‌دهد.
Human: من موسیقی دوست دارم.
